# Imports

In [2]:
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from typing import Callable

import kagglehub
from statsmodels.stats.outliers_influence import variance_inflation_factor

RANDOM_SEED = 42
np.random.seed = RANDOM_SEED

/home/janek/ds/2/aml/AML_project/.venv/lib/python3.14/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


# 1. Dataset preparation

## Downloads

In [3]:
path1 = kagglehub.dataset_download("shree0910/online-vs-in-store-shopping-behaviour-dataset")
path2 = kagglehub.dataset_download("vishardmehta/smartphone-battery-health-prediction-dataset")
path3 = kagglehub.dataset_download("foolishboi/blood-transfusion")
path4 = kagglehub.dataset_download("yasserh/wine-quality-dataset")

In [4]:
shopping_data = pd.read_csv(f"{path1}/{os.listdir(path1)[0]}")
smartphone_data = pd.read_csv(f"{path2}/{os.listdir(path2)[0]}")
smartphone_features = pd.read_csv(f"{path2}/{os.listdir(path2)[1]}")
blood_data = pd.read_csv(f"{path3}/{os.listdir(path3)[0]}")
wine_data = pd.read_csv(f"{path4}/{os.listdir(path4)[0]}")

## Preprocessing

### Functions

In [5]:
def cast_int(df, column):
    dict = {}
    i = 0
    for label in df[column].unique():
        dict[label] = i
        i += 1
    
    return df[column].map(dict)

In [6]:
def vifify(df, column):
    X = df.drop(column, axis=1)

    vif_data = pd.DataFrame()
    vif_data["feature"] = X.columns

    vif_data["VIF"] = [variance_inflation_factor(X.values, i)
                            for i in range(len(X.columns))]
    
    vif_data.sort_values(by="VIF", axis=0, ascending=False, inplace=True)
    
    return vif_data, X

In [7]:
def del_coll(df, col, threshold=5):
    column = col
    target = df[col]
    new_shop = df
    while True:
        vif, new_shop = vifify(new_shop, column)

        if vif.iloc[0,1] > threshold:
            column = vif.iloc[0,0]
        
        else:
            print(vif)
            break

    new_shop[col] = target
    
    return new_shop

### Shopping data

In [8]:
shopping_data["shopping_preference"] = cast_int(shopping_data, "shopping_preference")
shopping_data["city_tier"] = cast_int(shopping_data, "city_tier")
shopping_data["gender"] = cast_int(shopping_data, "gender")

In [9]:
shopping_data["shopping_preference"] = shopping_data["shopping_preference"].map({0: 0, 1: 1, 2:1})

In [10]:
shopping_data = del_coll(shopping_data, "shopping_preference")

                        feature       VIF
11            avg_delivery_days  4.768549
2            social_media_hours  4.728235
17          brand_loyalty_score  4.548954
0                monthly_income  4.505733
14  product_availability_online  4.479064
19          time_pressure_level  4.426047
4              tech_savvy_score  4.419042
15         impulse_buying_score  4.417325
16        need_touch_feel_score  4.406046
12     delivery_fee_sensitivity  4.405095
9          discount_sensitivity  4.402583
3    online_payment_trust_score  4.396035
18      environmental_awareness  4.393591
13       free_return_importance  4.371875
1        smartphone_usage_years  4.355952
8               avg_store_spend  3.879587
7              avg_online_spend  3.836766
5         monthly_online_orders  3.779715
6          monthly_store_visits  3.609372
10             return_frequency  3.281533
20                       gender  2.467068
21                    city_tier  2.450476


### Smartphone data

In [11]:
smartphone_data = smartphone_features.merge(smartphone_data, on="Device_ID").drop("current_battery_health_percent", axis=1)

In [12]:
smartphone_data["target_action"] = smartphone_data["recommended_action"].map({"Change Phone":1, "Replace Battery":1, "Keep Using":0})
smartphone_data = smartphone_data.drop(["recommended_action", "Device_ID"], axis=1)
smartphone_data["background_app_usage_level"] = cast_int(smartphone_data, "background_app_usage_level")
smartphone_data["signal_strength_avg"] = cast_int(smartphone_data, "signal_strength_avg")
smartphone_data.info()

<class 'pandas.DataFrame'>
RangeIndex: 5000 entries, 0 to 4999
Data columns (total 15 columns):
 #   Column                            Non-Null Count  Dtype  
---  ------                            --------------  -----  
 0   device_age_months                 5000 non-null   int64  
 1   battery_capacity_mah              5000 non-null   int64  
 2   avg_screen_on_hours_per_day       5000 non-null   float64
 3   avg_charging_cycles_per_week      5000 non-null   float64
 4   avg_battery_temp_celsius          5000 non-null   float64
 5   fast_charging_usage_percent       5000 non-null   float64
 6   overnight_charging_freq_per_week  5000 non-null   int64  
 7   gaming_hours_per_week             5000 non-null   float64
 8   video_streaming_hours_per_week    5000 non-null   float64
 9   background_app_usage_level        5000 non-null   int64  
 10  signal_strength_avg               5000 non-null   int64  
 11  charging_habit_score              5000 non-null   int64  
 12  usage_intensity_s

In [13]:
smartphone_data = del_coll(smartphone_data, "target_action", threshold=10)

                            feature       VIF
1      avg_charging_cycles_per_week  6.472615
2       fast_charging_usage_percent  5.237983
7               signal_strength_avg  4.501889
5    video_streaming_hours_per_week  4.358368
0                 device_age_months  3.542441
3  overnight_charging_freq_per_week  3.020644
4             gaming_hours_per_week  2.825954
6        background_app_usage_level  2.350005


In [14]:
smartphone_data[smartphone_data.duplicated()]

,device_age_months,avg_charging_cycles_per_week,fast_charging_usage_percent,overnight_charging_freq_per_week,gaming_hours_per_week,video_streaming_hours_per_week,background_app_usage_level,signal_strength_avg,target_action


### Blood data

In [15]:
blood_data.drop_duplicates(inplace=True, ignore_index=True)
blood_data.info()
blood_data["Class"] = blood_data["Class"].map({1:0, 2:1})

<class 'pandas.DataFrame'>
RangeIndex: 533 entries, 0 to 532
Data columns (total 5 columns):
 #   Column  Non-Null Count  Dtype
---  ------  --------------  -----
 0   V1      533 non-null    int64
 1   V2      533 non-null    int64
 2   V3      533 non-null    int64
 3   V4      533 non-null    int64
 4   Class   533 non-null    int64
dtypes: int64(5)
memory usage: 20.9 KB


In [16]:
blood_data = del_coll(blood_data, "Class", 10)

  feature       VIF
2      V4  5.550033
1      V3  3.558410
0      V1  2.201671


/home/janek/ds/2/aml/AML_project/.venv/lib/python3.14/site-packages/statsmodels/stats/outliers_influence.py:197: RuntimeWarning: divide by zero encountered in scalar divide
  vif = 1. / (1. - r_squared_i)


### Wine data

In [17]:
wine_data.info()
wine_data = wine_data.drop('Id', axis=1)

<class 'pandas.DataFrame'>
RangeIndex: 1143 entries, 0 to 1142
Data columns (total 13 columns):
 #   Column                Non-Null Count  Dtype  
---  ------                --------------  -----  
 0   fixed acidity         1143 non-null   float64
 1   volatile acidity      1143 non-null   float64
 2   citric acid           1143 non-null   float64
 3   residual sugar        1143 non-null   float64
 4   chlorides             1143 non-null   float64
 5   free sulfur dioxide   1143 non-null   float64
 6   total sulfur dioxide  1143 non-null   float64
 7   density               1143 non-null   float64
 8   pH                    1143 non-null   float64
 9   sulphates             1143 non-null   float64
 10  alcohol               1143 non-null   float64
 11  quality               1143 non-null   int64  
 12  Id                    1143 non-null   int64  
dtypes: float64(11), int64(2)
memory usage: 116.2 KB


In [18]:
wine_data["quality"].unique()

array([5, 6, 7, 4, 8, 3])

In [19]:
def cast_wine(df, column):
    dict = {}
    for label in df[column].unique():
        if label > 5:
            dict[label] = 1
        else:
            dict[label] = 0
    
    return df[column].map(dict)

wine_data["quality"] = cast_wine(wine_data, "quality")

In [20]:
wine_data = del_coll(wine_data, "quality", 10)

                feature       VIF
4   free sulfur dioxide  5.642393
0      volatile acidity  5.513306
5  total sulfur dioxide  5.490251
3             chlorides  4.906418
2        residual sugar  4.730015
1           citric acid  3.176401


### Assumed binary values for each 
- Shopping: 0 -- stationary,   1 -- online
- Smartphones: 0 -- no action,  1 -- replace battery or phone
- Blood: 0 -- did not donate,  1 -- donated
- Wine: 0 -- poor quality,  1 -- good quality

## Missing data generation

In [31]:
def remove_labels(df: pd.DataFrame, target: str, scheme: str) -> pd.DataFrame:
    """
    Remove labels in the dataset based on the chosen scheme. A new column named `target`_`scheme` is added to the original DataFrame.

    TODO: describe schemes

    MAR2: Firstly, all columns are normalized to interval [0, 1]. Then, for each row, the mean of values is calculated. The resulting series is categorized into 10 bins, of which an arbitrarily chosen bin (in this implementation, bin 3) is chosen. Each row belonging to this bin is removed. This results in 10% of missing labels.

    MNAR: This is analogous to MAR2 with one exception: if an observation belongs to class 1, the mean corresponding to it is transformed using formula x -> 1 - x.

    Parameters:
        df (pandas.DataFrame): A Dataframe containing a column in which the labels are to be removed.
        target (str): The name of the modified column.
        scheme (str): Scheme for generating missing labels: `"mcar"`, `"mar1"`, `"mar2"` or `"mnar"`. Providing other value will result in an error.

    Returns:
        df (pandas.DataFrame): The original DataFrame with a new column containing removed labels.
    """
    
    generator = np.random.default_rng(seed=RANDOM_SEED)
    if scheme == "mcar":
        pass
    elif scheme == "mar1":
        pass
    elif scheme == "mar2":
        normalized_value_means = df[df.columns.drop(list(df.filter(regex=target)))].apply(lambda x: (x.max() - x)/(x.max() - x.min()), axis=0).mean(axis=1)
        deciles = pd.qcut(normalized_value_means, 10, labels=False)
        is_missing = generator.binomial(1, deciles.map(lambda x: 0.9 if x == 3 else 0.05))
    elif scheme == "mnar":
        normalized_value_means = df[df.columns.drop(list(df.filter(regex=target)))].apply(lambda x: (x.max() - x)/(x.max() - x.min()), axis=0).mean(axis=1)
        normalized_value_means[df[target] == 1] = 1 - normalized_value_means[df[target] == 1]
        deciles = pd.qcut(normalized_value_means, 10, labels=False)
        is_missing = generator.binomial(1, deciles.map(lambda x: 0.9 if x == 3 else 0.05))
    else:
        raise ValueError("Argument scheme accepts only one of following values: 'mcar', 'mar1', 'mar2' or 'mnar'.")
    print(f"Percentage of labels removed: {is_missing.mean()*100:.2f}%")
    df[target + "_" + scheme] = df[target]
    df.loc[is_missing, target + "_" + scheme] = -1
    return df

### MCAR

In [22]:
# Missing Completely At Random scheme
# Probability of a missing label is constant

# TODO: fill the code in the function above

### MAR 1

In [23]:
# Missing At Random 1 scheme
# Probability of a missing label is determined based on one feature

# TODO: fill the code in the function above

### MAR 2

In [32]:
shopping_data = remove_labels(shopping_data, "shopping_preference", "mar2")
smartphone_data = remove_labels(smartphone_data, "target_action", "mar2")
blood_data = remove_labels(blood_data, "Class", "mar2")
wine_data = remove_labels(wine_data, "quality", "mar2")

Percentage of labels removed: 13.83%
Percentage of labels removed: 13.26%
Percentage of labels removed: 13.32%
Percentage of labels removed: 13.04%


### MNAR

In [33]:
shopping_data = remove_labels(shopping_data, "shopping_preference", "mnar")
smartphone_data = remove_labels(smartphone_data, "target_action", "mnar")
blood_data = remove_labels(blood_data, "Class", "mnar")
wine_data = remove_labels(wine_data, "quality", "mnar")

Percentage of labels removed: 13.59%
Percentage of labels removed: 13.40%
Percentage of labels removed: 12.95%
Percentage of labels removed: 14.09%
